In [1]:
import cv2
import numpy as np
import time
import json
import queue
from typing import List, Optional, Literal
from tensorflow.keras.models import Model, load_model
from landmarkers.inferences import InferenceSequence, Inference
from landmarkers.landmarks import LandmarksSequence
from landmarkers.mp.hands import MediapipeHandsMetadata, MPLiveStreamLandmarker

# -------------------------
# Cargar configuración desde config.json
# -------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

# Datos comunes
common_config = config['common']
ACTIONS: np.ndarray = np.array(common_config['actions'])
SEQUENCE_LENGTH: int = common_config['sequence_length']

# Datos específicos de train_model (test_model usa la misma configuración)
train_config = config['train_model']
MODEL_EXPORT_NAME: str = train_config['model_export_name']
HAND_SELECTION: str = train_config['hand_selection']

# Parámetros de inferencia (no están en config, se mantienen fijos)
THRESHOLD: float = 0.85
PRED_BUFFER_SIZE: int = 10
NUM_LANDMARKS: int = 21

# -------------------------
# Carga de modelo y secuencias
# -------------------------
model: Model = load_model(MODEL_EXPORT_NAME)
seq_right: InferenceSequence = InferenceSequence(fixed_buffer_length=SEQUENCE_LENGTH)
seq_left: InferenceSequence = InferenceSequence(fixed_buffer_length=SEQUENCE_LENGTH)
pred_buffer: List[np.ndarray] = []

# -------------------------
# Funciones auxiliares
# -------------------------
def get_timestamp_ms() -> int:
	return int(time.time() * 1000)

# ------------------------------------------------------------
# Funcion sigmoide
# ------------------------------------------------------------
def sigmoid(x):
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )

# ------------------------------------------------------------
# Obtener velocidades del centroide normalizadas
# ------------------------------------------------------------
def get_centroid_velocity_norm_sequence(landmark_sequence) -> np.ndarray:
    """
    Devuelve un array (T, 3) con la velocidad del centroide
    normalizada mediante sigmoide.
    """
    time_stamps = np.array(landmark_sequence.time_stamps_ms, dtype=np.float32)
    centroids = np.array(landmark_sequence.centroid(), dtype=np.float32)
    d_centroids = np.diff(centroids, axis=0)          # (T-1, 3)
    d_time = np.diff(time_stamps)[:, None]            # (T-1, 1)
    d_time[d_time == 0] = 1e-6
    velocity = d_centroids / d_time                   # (T-1, 3)
    norms = np.linalg.norm(velocity, axis=1, keepdims=True)
    norms[norms == 0] = 1e-6
    norm_velocity = (sigmoid(norms) / norms) * velocity
    norm_velocity = np.vstack([
        np.zeros((1, 3), dtype=np.float32),
        norm_velocity
    ])

    return norm_velocity.astype(np.float32)

def preprocess_sequence(right_seq: InferenceSequence, left_seq: InferenceSequence) -> np.ndarray:
    """
    Concatena landmarks de ambas manos frame a frame
    - Resamplea cada mano a SEQUENCE_LENGTH frames
    - Centra cada frame respecto al centroid
    - Devuelve np.ndarray de shape (SEQUENCE_LENGTH, features) listo para el modelo
    """
    
    r_landmark_sequence: LandmarksSequence = right_seq.landmarks_sequence
    r_landmark_sequence = r_landmark_sequence.resample()

    # Vector de velocidad del centroide
    r_centroid_velocity_norm_sequence = get_centroid_velocity_norm_sequence(r_landmark_sequence)  # (T,3)

    # Centrar landmarks
    r_landmark_sequence = r_landmark_sequence.centered(0)
    
    l_landmark_sequence: LandmarksSequence = left_seq.landmarks_sequence
    l_landmark_sequence = l_landmark_sequence.resample()

    # Vector de velocidad del centroide
    l_centroid_velocity_norm_sequence = get_centroid_velocity_norm_sequence(l_landmark_sequence)  # (T,3)

    # Centrar landmarks
    l_landmark_sequence = l_landmark_sequence.centered(0)
    
    r_landmarks_array = r_landmark_sequence.array
    l_landmarks_array = l_landmark_sequence.array 

    # Convertir a arrays
    r_array = np.concatenate(
        [r_landmarks_array, r_centroid_velocity_norm_sequence[:, None, :]],  # (T,1,3)
        axis=1
    )  # shape (n_frames, 21, 3)
    l_array = np.concatenate(
        [l_landmarks_array, l_centroid_velocity_norm_sequence[:, None, :]],  # (T,1,3)
        axis=1
    ) # shape (n_frames, 21, 3)

    # Interpolación simple si los frames no son SEQUENCE_LENGTH
    n_frames = SEQUENCE_LENGTH
    if r_array.shape[0] != n_frames:
        idxs = np.linspace(0, r_array.shape[0]-1, n_frames, dtype=int)
        r_array = r_array[idxs]
    if l_array.shape[0] != n_frames:
        idxs = np.linspace(0, l_array.shape[0]-1, n_frames, dtype=int)
        l_array = l_array[idxs]

    # Seleccionar mano(s) según configuración
    if HAND_SELECTION == 'left':
        # Solo mano izquierda
        frames = []
        for i in range(n_frames):
            l_frame = l_array[i].reshape(-1)  # (21*3=63)
            frames.append(l_frame)
        return np.array(frames, dtype=np.float32)  # (SEQUENCE_LENGTH, 63)
    elif HAND_SELECTION == 'right':
        # Solo mano derecha
        frames = []
        for i in range(n_frames):
            r_frame = r_array[i].reshape(-1)  # (21*3=63)
            frames.append(r_frame)
        return np.array(frames, dtype=np.float32)  # (SEQUENCE_LENGTH, 63)
    else:  # 'both'
        # Ambas manos
        frames = []
        for i in range(n_frames):
            r_frame = r_array[i].reshape(-1)  # (21*3=63)
            l_frame = l_array[i].reshape(-1)  # (21*3=63)
            frames.append(np.hstack([r_frame, l_frame]))  # (126,)
        return np.array(frames, dtype=np.float32)  # (SEQUENCE_LENGTH, 126)

def predict_gesture(features: np.ndarray, model: Model, pred_buffer: List[np.ndarray], threshold: float = THRESHOLD) -> Optional[str]:
	features = np.expand_dims(features, axis=0)  # agregar batch
	res = model.predict(features, verbose=0)[0]
	pred_buffer.append(res)
	avg = np.mean(pred_buffer[-PRED_BUFFER_SIZE:], axis=0)
	idx = int(np.argmax(avg))
	if avg[idx] > threshold and ACTIONS[idx] != "none":
		return str(ACTIONS[idx])
	return None

def get_hand(inferences: Optional[List[Inference]], category: Literal["Left", "Right"]) -> Inference:
	"""Devuelve la inferencia de la mano o una vacía si no está"""
	if inferences is None:
		zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
		return Inference(zeros, zeros, MediapipeHandsMetadata(category_name=category, index=0, score=0.0))
	hands = [h for h in inferences if h.metadata.category_name == category]
	if hands:
		return hands[0]
	zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
	return Inference(zeros, zeros, MediapipeHandsMetadata(category_name=category, index=0, score=0.0))

def render_frame(frame: np.ndarray, right_hand: Inference, left_hand: Inference, gesture: Optional[str] = None) -> np.ndarray:
	"""Dibuja landmarks de ambas manos y la acción detectada"""
	for lm in right_hand.landmarks.array:
		x, y = int(lm[0]*frame.shape[1]), int(lm[1]*frame.shape[0])
		cv2.circle(frame, (x, y), 3, (0,255,0), -1)
	for lm in left_hand.landmarks.array:
		x, y = int(lm[0]*frame.shape[1]), int(lm[1]*frame.shape[0])
		cv2.circle(frame, (x, y), 3, (0,0,255), -1)
	if gesture:
		cv2.putText(frame, gesture, (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255,0), 3)
	return frame

# -------------------------
# Cola de frames
# -------------------------
frame_queue: "queue.Queue[tuple[np.ndarray, List[Inference], int]]" = queue.Queue(maxsize=1)

def stream_callback(inferences: List[Inference], frame: np.ndarray, timestamp_ms: int) -> None:
	try:
		frame_queue.put_nowait((frame, inferences, timestamp_ms))
	except queue.Full:
		pass

# -------------------------
# Loop principal con MPLiveStreamLandmarker
# -------------------------
cap = cv2.VideoCapture(0)
with MPLiveStreamLandmarker(model_path="hand_landmarker.task", callback=stream_callback, num_hands=2) as live_landmarker:
	while cap.isOpened():
		ret, frame = cap.read()
		if not ret:
			break

		ts = get_timestamp_ms()
		live_landmarker.infer(frame, ts)

		if not frame_queue.empty():
			f, inferences, ts = frame_queue.get()

			right_hand = get_hand(inferences, "Right")
			left_hand = get_hand(inferences, "Left")

			# Append a las secuencias individuales
			seq_right.append(right_hand, ts)
			seq_left.append(left_hand, ts)

			gesture: Optional[str] = None
			if len(seq_right) == SEQUENCE_LENGTH and len(seq_left) == SEQUENCE_LENGTH:
				features = preprocess_sequence(seq_right, seq_left)  # shape (SEQUENCE_LENGTH, features)
				gesture = predict_gesture(features, model, pred_buffer)

			show_frame = render_frame(f, right_hand, left_hand, gesture)
			cv2.imshow("Feed", show_frame)

		if cv2.waitKey(10) & 0xFF == ord("q"):
			break

cap.release()
cv2.destroyAllWindows()


2026-04-02 16:55:08.094935: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-02 16:55:08.149584: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-02 16:55:09.483643: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
2026-04-02 16:55:10.989178: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA erro